In [1]:
from dotenv import load_dotenv
from langchain_community.embeddings import DashScopeEmbeddings
from pymilvus import MilvusClient, AnnSearchRequest, Function, FunctionType

from app.core.config import settings

# 准备工作
load_dotenv()
client = MilvusClient(uri=settings.rag.milvus_url)
embedding = DashScopeEmbeddings(model="text-embedding-v4")

# 定义搜索功能
def hybrid_search(query: str, ranker, filter_query: str|None = None):
    # 准备稠密向量检索条件
    query_vector = embedding.embed_query(query)
    dense_request = AnnSearchRequest(
        data=[query_vector],
        anns_field='dense',
        param={'metric_type':'COSINE'},
        limit=3,
        filter=filter_query
    )

    # 准备稀疏向量检索条件
    sparse_request = AnnSearchRequest(
        data=[query],
        anns_field='sparse',
        param={'metric_type':'BM25'},
        limit=3,
        filter=filter_query
    )

    # 准备混合检索条件
    requests = [dense_request, sparse_request]

    # 设置整合策略并检索
    res = client.hybrid_search(
        collection_name='my_1',
        reqs=requests, #混合检索条件，包括稠密向量检索条件和稀疏向量检索条件
        ranker=ranker, #融合排序策略
        limit=2,
        output_fields=['h2', 'text']
    )
    # 打印检索结果
    for hits in res:
        for hit in hits:
            print(hit)

# 加权融合策略（从pymilvus中导入WeightedRanker）
# ranker = WeightedRanker(0.5, 0.5)
# hybrid_search("教育的负面作用是什么", ranker)

# RRF融合策略（从pymilvus中导入RRFRanker）
# ranker = RRFRanker(k=60)
# hybrid_search("教育的负面作用是什么", ranker)

# 定义Cross-Encoder
def create_cross_encoder_ranker(queries: list[str]):
    return Function(
        name="dashscope_semantic_ranker", #ranker名称，唯一即可
        input_field_names=['text'], #原始文档字段
        function_type=FunctionType.RERANK, #固定值
        params={
            "reranker":"model", #基于模型rerank，即cross-encoder
            "provider": "ali", #rerank模型提供者，阿里
            "model_name": "gte-rerank-v2", #rerank模型名称
            "queries": queries, #查询条件
            "max_client_batch_size": 5 #单批次中处理的文档最大数量
        }
    )


query = "教育的负面作用是什么"
# 定义Cross-encoder对象
reranker = create_cross_encoder_ranker([query])
# 实现混合检索
hybrid_search(query, reranker,filter="h2 == '第一节 教育的功能'")

C:\Users\Administrator\AppData\Local\Temp\ipykernel_20144\929264601.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.embeddings import DashScopeEmbeddings


{'id': 15, 'distance': 0.29470494389533997, 'entity': {'text': '# 第二章 教育基本原理  \n## 第一节 教育的功能  \n### （一）个体发展功能和社会发展功能  \n教育的正向功能（积极功能）指教育有助于社会进步和个体发展的积极影响和作用。  \n教育的负向功能（消极功能）指阻碍社会进步和个体发展的消极影响和作用。  \n教育的显性功能是指教育活动依照教育目的，在实际运行中所出现的与之相吻合的结果。  \n教育的隐性功能指伴随显性功能所出现的非预期性的功能。', 'h2': '第一节 教育的功能'}}
{'id': 21, 'distance': 0.1374114453792572, 'entity': {'text': '### （六）生产力与教育的关系  \n生产力对其它一切因素都起着决定的作用，是决定教育性质的根本因素。  \n生产力对教育的主要作用表现为：生产力发展水平决定着教育事业发展的规模和速度；生产力发展水平制约着人才培养的规格与教育结构；生产力的发展促进教育内容、教学方法和教学组织形式的发展与改革。  \n教育对生产力的作用表现为：教育再生产劳动力；教育是科学知识与技术发展的重要手段。', 'h2': '第二节 教育和社会的关系'}}
